In [1]:
import re
import regex
assert "regex" in str(regex.compile("(test){e<=1}"))
import pandas as pd
import os
import numpy as np
pd.set_option('display.max_colwidth', 80)
from nltk.corpus import stopwords
from sklearn.model_selection import (train_test_split, StratifiedKFold,
cross_val_predict)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from pathlib import Path
project_dir = Path().resolve()
os.chdir(project_dir)

In [2]:
df = pd.read_csv('hand_coded_sample_post_cor.csv')
print(len(df))
for col in ["complaint_clean", "findings_clean"]:
    df[col] = (
        df[col]
        .str.replace(r"\bcpoa\b", "agency", regex=True, case=False)
        .str.replace(r"\bapd\b", "police", regex=True, case=False)
    )
df['snippet_clean'] = (
    df["complaint_clean"].fillna("").str.strip()
    + "\n"
    + df["findings_clean"].fillna("").str.strip()
).str.strip()

190


In [3]:
df[[
    "regulatory_complaint",
    "demeanor_complaint",
    "bias_complaint",
    "traffic_scenario",
    "property_complaint"
]].apply(pd.Series.value_counts)

,regulatory_complaint,demeanor_complaint,bias_complaint,traffic_scenario,property_complaint
0,154,120,179,156,176
1,36,70,11,34,14


In [4]:
vectorizers = {
    "tfidf": TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.80
    ),

    "sbert": SentenceTransformer("all-MiniLM-L6-v2")
}

models = {
    "nb_bal": MultinomialNB(class_prior=[0.5,0.5]
    ),

    "logreg_2k_bal": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=123
    ),

    "logreg_1k_bal": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=123
    ),

    "svm_lin": LinearSVC(
        class_weight="balanced",
        random_state=123
    ),

    "rf": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=123,
        n_jobs=-1
    ),

    "svm": SVC(
    kernel="linear",
    class_weight="balanced",
    random_state=123
)

}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## MH SNIPPET

In [5]:
TARGET = "mental_health"
TEXT_COLUMN = "snippet_clean"

In [6]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       180
           1       0.00      0.00      0.00        10

    accuracy                           0.95       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.90      0.95      0.92       190

[[180   0]
 [ 10   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       180
           1       0.50      0.10      0.17        10

    accuracy                           0.95       190
   macro avg       0.73      0.55      0.57       190
weighted avg       0.93      0.95      0.93       190

[[179   1]
 [  9   1]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       180
           1       0.50      0.10      0.17        10

    accuracy                           0.95       190
   macro avg       0.73      0.55      0.57       190
weighted avg       0.93      0.95      0.93       190

[[179   1]
 [  9   1]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : sv

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : mental_health
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.97      0.94      0.95       180
           1       0.27      0.40      0.32        10

    accuracy                           0.91       190
   macro avg       0.62      0.67      0.64       190
weighted avg       0.93      0.91      0.92       190

[[169  11]
 [  6   4]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.97      0.94      0.95       180
           1       0.27      0.40      0.32        10

    accuracy                           0.91       190
   macro avg       0.62      0.67      0.64       190
weighted avg       0.93      0.91      0.92       190

[[169  11]
 [  6   4]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : sbert
Model       : sv

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## MH COMPLAINT

In [7]:
TARGET = "mental_health"
TEXT_COLUMN = "complaint_clean"

In [8]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       180
           1       0.00      0.00      0.00        10

    accuracy                           0.95       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.90      0.95      0.92       190

[[180   0]
 [ 10   0]]
Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       180
           1       0.00      0.00      0.00        10

    accuracy                           0.95       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.90      0.95      0.92       190

[[180   0]
 [ 10   0]]
Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : log

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm_lin
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       180
           1       0.00      0.00      0.00        10

    accuracy                           0.95       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.90      0.95      0.92       190

[[180   0]
 [ 10   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       180
           1       0.00      0.00      0.00        10

    accuracy                           0.95       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.90      0.95      0.92       190

[[180   0]
 [ 10   0]]
Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       180
           1       0.00      0.00      0.00        10

    accuracy                           0.95       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.90      0.95      0.92       190

[[180   0]
 [ 10   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : mental_health
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.96      0.91      0.94       180
           1       0.20      0.40      0.27        10

    accuracy                           0.88       190
   macro avg       0.58      0.66      0.60       190
weighted avg       0.92      0.88      0.90       190

[[164  16]
 [  6   4]]
Target      : mental_health
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.96      0.91      0.94       180
           1       0.20      0.40      0.27        10

    accuracy                           0.88       190
   macro avg       0.58      0.66      0.60       190
weighted avg       0.92      0.88      0.90       190

[[164  16]
 [  6   4]]
Target      : mental_health
Text        : complaint_clean
Vectorizer  : sbert
Model     

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## BIAS SNIPPET

In [9]:
TARGET = "bias_complaint"
TEXT_COLUMN = "snippet_clean"

In [10]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm_lin
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       179
           1       0.40      0.55      0.46        11

    accuracy                           0.93       190
   macro avg       0.69      0.75      0.71       190
weighted avg       0.94      0.93      0.93       190

[[170   9]
 [  5   6]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.97      0.95      0.96       179
           1       0.40      0.55      0.46        11

    accuracy                           0.93       190
   macro avg       0.69      0.75      0.71       190
weighted avg       0.94      0.93      0.93       190

[[170   9]
 [  5   6]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       :

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## BIAS COMPLAINT

In [11]:
TARGET = "bias_complaint"
TEXT_COLUMN = "complaint_clean"

In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]
Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]
Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : 

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]
Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       179
           1       0.00      0.00      0.00        11

    accuracy                           0.94       190
   macro avg       0.47      0.50      0.49       190
weighted avg       0.89      0.94      0.91       190

[[179   0]
 [ 11   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       179
           1       0.44      0.36      0.40        11

    accuracy                           0.94       190
   macro avg       0.70      0.67      0.68       190
weighted avg       0.93      0.94      0.93       190

[[174   5]
 [  7   4]]
Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       179
           1       0.44      0.36      0.40        11

    accuracy                           0.94       190
   macro avg       0.70      0.67      0.68       190
weighted avg       0.93      0.94      0.93       190

[[174   5]
 [  7   4]]
Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model  

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## PROPERTY SNIPPET

In [13]:
TARGET = "property_complaint"
TEXT_COLUMN = "snippet_clean"

In [14]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       176
           1       0.00      0.00      0.00        14

    accuracy                           0.93       190
   macro avg       0.46      0.50      0.48       190
weighted avg       0.86      0.93      0.89       190

[[176   0]
 [ 14   0]]
Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       176
           1       0.00      0.00      0.00        14

    accuracy                           0.92       190
   macro avg       0.46      0.50      0.48       190
weighted avg       0.86      0.92      0.89       190

[[175   1]
 [ 14   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       176
           1       0.00      0.00      0.00        14

    accuracy                           0.92       190
   macro avg       0.46      0.50      0.48       190
weighted avg       0.86      0.92      0.89       190

[[175   1]
 [ 14   0]]
Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm_lin
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       176
           1       0.50      0.07      0.12        14

    accuracy                           0.93       190
   macro avg       0.72      0.53      0.54       190
weighted avg       0.90      0.93      0.90       190

[[175   1]
 [ 13   1]]
Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model  

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : property_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       176
           1       0.42      0.57      0.48        14

    accuracy                           0.91       190
   macro avg       0.69      0.75      0.72       190
weighted avg       0.92      0.91      0.92       190

[[165  11]
 [  6   8]]
Target      : property_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       176
           1       0.42      0.57      0.48        14

    accuracy                           0.91       190
   macro avg       0.69      0.75      0.72       190
weighted avg       0.92      0.91      0.92       190

[[165  11]
 [  6   8]]
Target      : property_complaint
Text        : snippet_clean
Vectorizer  : sbert
M

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## PROPERTY COMPLAINT

In [15]:
TARGET = "property_complaint"
TEXT_COLUMN = "complaint_clean"

In [16]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       176
           1       0.00      0.00      0.00        14

    accuracy                           0.93       190
   macro avg       0.46      0.50      0.48       190
weighted avg       0.86      0.93      0.89       190

[[176   0]
 [ 14   0]]
Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.93      0.98      0.96       176
           1       0.25      0.07      0.11        14

    accuracy                           0.92       190
   macro avg       0.59      0.53      0.53       190
weighted avg       0.88      0.92      0.89       190

[[173   3]
 [ 13   1]]
Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Mo

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       176
           1       0.00      0.00      0.00        14

    accuracy                           0.93       190
   macro avg       0.46      0.50      0.48       190
weighted avg       0.86      0.93      0.89       190

[[176   0]
 [ 14   0]]
Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.93      0.98      0.95       176
           1       0.00      0.00      0.00        14

    accuracy                           0.91       190
   macro avg       0.46      0.49      0.48       190
weighted avg       0.86      0.91      0.88       190

[[173   3]
 [ 14   0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : property_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       176
           1       0.13      0.14      0.14        14

    accuracy                           0.87       190
   macro avg       0.53      0.53      0.53       190
weighted avg       0.87      0.87      0.87       190

[[163  13]
 [ 12   2]]
Target      : property_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       176
           1       0.13      0.14      0.14        14

    accuracy                           0.87       190
   macro avg       0.53      0.53      0.53       190
weighted avg       0.87      0.87      0.87       190

[[163  13]
 [ 12   2]]
Target      : property_complaint
Text        : complaint_clean
Vectorizer  : s

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## DEMEANOR SNIPPET

In [17]:
TARGET = "demeanor_complaint"
TEXT_COLUMN = "snippet_clean"

In [18]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.64      0.97      0.77       120
           1       0.50      0.06      0.10        70

    accuracy                           0.63       190
   macro avg       0.57      0.51      0.44       190
weighted avg       0.59      0.63      0.52       190

[[116   4]
 [ 66   4]]
Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.74      0.72      0.73       120
           1       0.54      0.56      0.55        70

    accuracy                           0.66       190
   macro avg       0.64      0.64      0.64       190
weighted avg       0.67      0.66      0.66       190

[[87 33]
 [31 39]]
Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.77      0.72      0.74       120
           1       0.56      0.63      0.59        70

    accuracy                           0.68       190
   macro avg       0.67      0.67      0.67       190
weighted avg       0.69      0.68      0.69       190

[[86 34]
 [26 44]]
Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.77      0.72      0.74       120
           1       0.56      0.63      0.59        70

    accuracy                           0.68       190
   macro avg       0.67      0.67      0.67       190
weighted avg       0.69      0.68      0.69       190

[[86 34]
 [26 44]]
Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model    

## DEMEANOR COMPLAINT

In [19]:
TARGET = "demeanor_complaint"
TEXT_COLUMN = "complaint_clean"

In [20]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.67      0.94      0.78       120
           1       0.67      0.20      0.31        70

    accuracy                           0.67       190
   macro avg       0.67      0.57      0.54       190
weighted avg       0.67      0.67      0.61       190

[[113   7]
 [ 56  14]]
Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.74      0.72      0.73       120
           1       0.54      0.56      0.55        70

    accuracy                           0.66       190
   macro avg       0.64      0.64      0.64       190
weighted avg       0.67      0.66      0.66       190

[[87 33]
 [31 39]]
Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model 

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.80      0.75      0.78       120
           1       0.62      0.69      0.65        70

    accuracy                           0.73       190
   macro avg       0.71      0.72      0.71       190
weighted avg       0.73      0.73      0.73       190

[[90 30]
 [22 48]]
Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.80      0.75      0.78       120
           1       0.62      0.69      0.65        70

    accuracy                           0.73       190
   macro avg       0.71      0.72      0.71       190
weighted avg       0.73      0.73      0.73       190

[[90 30]
 [22 48]]
Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : sbert
Mod

## REGULATORY SNIPPET

In [21]:
TARGET = "regulatory_complaint"
TEXT_COLUMN = "snippet_clean"

In [22]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.81      1.00      0.90       154
           1       0.00      0.00      0.00        36

    accuracy                           0.81       190
   macro avg       0.41      0.50      0.45       190
weighted avg       0.66      0.81      0.73       190

[[154   0]
 [ 36   0]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.88      0.95      0.92       154
           1       0.68      0.47      0.56        36

    accuracy                           0.86       190
   macro avg       0.78      0.71      0.74       190
weighted avg       0.85      0.86      0.85       190

[[146   8]
 [ 19  17]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Mo

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm_lin
              precision    recall  f1-score   support

           0       0.87      0.97      0.91       154
           1       0.72      0.36      0.48        36

    accuracy                           0.85       190
   macro avg       0.79      0.66      0.70       190
weighted avg       0.84      0.85      0.83       190

[[149   5]
 [ 23  13]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.81      1.00      0.90       154
           1       0.00      0.00      0.00        36

    accuracy                           0.81       190
   macro avg       0.41      0.50      0.45       190
weighted avg       0.66      0.81      0.73       190

[[154   0]
 [ 36   0]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.85      0.75      0.80       154
           1       0.29      0.44      0.35        36

    accuracy                           0.69       190
   macro avg       0.57      0.60      0.57       190
weighted avg       0.75      0.69      0.71       190

[[115  39]
 [ 20  16]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.85      0.75      0.80       154
           1       0.29      0.44      0.35        36

    accuracy                           0.69       190
   macro avg       0.57      0.60      0.57       190
weighted avg       0.75      0.69      0.71       190

[[115  39]
 [ 20  16]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : s

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## REGULATORY COMPLAINT

In [23]:
TARGET = "regulatory_complaint"
TEXT_COLUMN = "complaint_clean"

In [24]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.81      0.99      0.89       154
           1       0.50      0.03      0.05        36

    accuracy                           0.81       190
   macro avg       0.66      0.51      0.47       190
weighted avg       0.75      0.81      0.74       190

[[153   1]
 [ 35   1]]
Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.86      0.94      0.90       154
           1       0.59      0.36      0.45        36

    accuracy                           0.83       190
   macro avg       0.73      0.65      0.67       190
weighted avg       0.81      0.83      0.81       190

[[145   9]
 [ 23  13]]
Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : tf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.88      0.78      0.82       154
           1       0.36      0.53      0.43        36

    accuracy                           0.73       190
   macro avg       0.62      0.65      0.63       190
weighted avg       0.78      0.73      0.75       190

[[120  34]
 [ 17  19]]
Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.88      0.78      0.82       154
           1       0.36      0.53      0.43        36

    accuracy                           0.73       190
   macro avg       0.62      0.65      0.63       190
weighted avg       0.78      0.73      0.75       190

[[120  34]
 [ 17  19]]
Target      : regulatory_complaint
Text        : complaint_clean
Vectorize

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## TRAFFIC SNIPPET

In [25]:
TARGET = "traffic_scenario"
TEXT_COLUMN = "snippet_clean"

In [26]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.83      1.00      0.90       156
           1       1.00      0.03      0.06        34

    accuracy                           0.83       190
   macro avg       0.91      0.51      0.48       190
weighted avg       0.86      0.83      0.75       190

[[156   0]
 [ 33   1]]
Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.89      0.96      0.92       156
           1       0.70      0.47      0.56        34

    accuracy                           0.87       190
   macro avg       0.79      0.71      0.74       190
weighted avg       0.86      0.87      0.86       190

[[149   7]
 [ 18  16]]
Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : tfidf
Model       : 

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.94      0.89      0.91       156
           1       0.60      0.74      0.66        34

    accuracy                           0.86       190
   macro avg       0.77      0.81      0.79       190
weighted avg       0.88      0.86      0.87       190

[[139  17]
 [  9  25]]
Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.94      0.89      0.91       156
           1       0.60      0.74      0.66        34

    accuracy                           0.86       190
   macro avg       0.77      0.81      0.79       190
weighted avg       0.88      0.86      0.87       190

[[139  17]
 [  9  25]]
Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : sbert
Model  

## TRAFFIC COMPLAINT

In [27]:
TARGET = "traffic_scenario"
TEXT_COLUMN = "complaint_clean"

In [28]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

for vectorizer_name, vectorizer in vectorizers.items():
    # TF-IDF
    if vectorizer_name == "tfidf":
        for model_name, clf in models.items():
            pipe = Pipeline([
                ("vectorizer", vectorizer),
                ("classifier", clf)])

            y_pred = cross_val_predict(
                pipe,
                X,
                y,
                cv=cv,
                n_jobs=-1)

            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)

            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))
    # S-Bert
    elif vectorizer_name == "sbert":
        X_vec = vectorizer.encode(
            X.tolist(),
            show_progress_bar=True)
        for model_name, clf in models.items():
            if model_name in ["nb_bal", "nb_unbal"]: # no doing nb for sbert
                continue
            y_pred = cross_val_predict(
                clf,
                X_vec,
                y,
                cv=cv,
                n_jobs=-1
            )
            print("=" * 70)
            print(f"Target      : {TARGET}")
            print(f"Text        : {TEXT_COLUMN}")
            print(f"Vectorizer  : {vectorizer_name}")
            print(f"Model       : {model_name}")
            print("=" * 70)
            print(classification_report(y, y_pred))
            print(confusion_matrix(y, y_pred))

Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.84      1.00      0.91       156
           1       1.00      0.15      0.26        34

    accuracy                           0.85       190
   macro avg       0.92      0.57      0.59       190
weighted avg       0.87      0.85      0.80       190

[[156   0]
 [ 29   5]]
Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.88      0.91      0.90       156
           1       0.52      0.44      0.48        34

    accuracy                           0.83       190
   macro avg       0.70      0.68      0.69       190
weighted avg       0.82      0.83      0.82       190

[[142  14]
 [ 19  15]]
Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : tfidf
Model   

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.95      0.90      0.92       156
           1       0.62      0.76      0.68        34

    accuracy                           0.87       190
   macro avg       0.78      0.83      0.80       190
weighted avg       0.89      0.87      0.88       190

[[140  16]
 [  8  26]]
Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.95      0.90      0.92       156
           1       0.62      0.76      0.68        34

    accuracy                           0.87       190
   macro avg       0.78      0.83      0.80       190
weighted avg       0.89      0.87      0.88       190

[[140  16]
 [  8  26]]
Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : sbert
M